# Compare models

Reads every run folder under `models/` and plots them side by side. A run folder is any folder with a
`metrics.json` (folders named `seed_*` are the seeds of their parent run). Train runs with a name so they
don't overwrite each other:

```
python -m engines.neural_network.train --name more_dropout      # -> models/neural_network/more_dropout/
```

**Adding a baseline** (median, linear, LightGBM, ...): split with `engines.common.load_splits(...)` using the
same seeds as the config, score with `engines.common.regression_metrics`, and write the same files:

| file | content | needed for |
|---|---|---|
| `seed_<s>/metrics.json` | `{split: {"overall": {...}, "<type>": {..., "n": n}}}` | everything |
| `seed_<s>/test_predictions.csv` | `property_type, district, actual, predicted` | error plots |
| `seed_<s>/history.csv` | `epoch, train_loss, val_loss` | learning curves (optional) |

Compare **test** numbers only, and only between runs that used the same seeds.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import NullFormatter

LAB = Path.cwd() if (Path.cwd() / "engines").exists() else Path.cwd().parent
MODELS_DIR = Path(os.environ.get("MODELS_DIR", LAB / "models"))

FOCUS_RUN = None  # run shown in the detail plots; None = best test MdAPE among runs with the most seeds
REF_RUN = None    # run the others are compared against; same default
MIN_ROWS = 10     # property types with fewer test rows are left out of the detail plots
METRICS = ["median_ape", "rmse_log", "r2", "mape", "mae", "rmse"]
LOWER_BETTER = {"median_ape": True, "rmse_log": True, "r2": False, "mape": True, "mae": True, "rmse": True}
LABEL = {"median_ape": "MdAPE %", "rmse_log": "RMSE log", "r2": "R²", "mape": "MAPE %", "mae": "MAE $", "rmse": "RMSE $"}

# palette (light mode): categorical slots in fixed order, sequential blue ramp, chart chrome
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK2, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.edgecolor": AXIS,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "lines.linewidth": 2, "legend.frameon": False,
    "figure.dpi": 110,
})
print("models dir:", MODELS_DIR)

## Load runs

In [ ]:
def seed_dirs(run_dir):
    """[(seed, folder)] for a run: its seed_* folders, else the run folder itself (seed unknown)."""
    seeds = sorted((int(p.name[5:]), p) for p in run_dir.glob("seed_*") if (p / "metrics.json").exists())
    return seeds or [(None, run_dir)]

runs = {}
for f in sorted(MODELS_DIR.rglob("metrics.json")):
    d = f.parent
    if not d.name.startswith("seed_"):
        runs[d.relative_to(MODELS_DIR).as_posix()] = d

rows = []
for run, d in runs.items():
    for seed, sd in seed_dirs(d):
        for split, groups in json.load(open(sd / "metrics.json")).items():
            for group, m in groups.items():
                rows += [{"run": run, "seed": seed, "split": split, "group": group, "metric": k,
                          "value": m[k], "n": m.get("n")} for k in METRICS if k in m]
scores = pd.DataFrame(rows)
if scores.empty:
    raise SystemExit(f"no runs under {MODELS_DIR} - train one first")
test = scores[scores.split == "test"]
overall = test[test.group == "overall"]

order = (overall[overall.metric == "median_ape"].groupby("run").value.mean().sort_values().index.tolist())
n_seeds = overall[overall.metric == "r2"].groupby("run").size()
best = next(r for r in order if n_seeds[r] == n_seeds.max())  # 1-seed runs are too noisy to be the default
FOCUS = FOCUS_RUN or best
REF = REF_RUN or best
print(f"{len(runs)} runs; focus = {FOCUS}; reference = {REF}")

## Test scores (mean ± std over seeds)
Sorted by test MdAPE. `seeds` = how many splits each number averages.

In [ ]:
def pm(v):
    m, s = v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)
    return f"{m:,.0f} ± {s:,.0f}" if abs(m) >= 1000 else f"{m:.3f} ± {s:.3f}"

table = overall.pivot_table(index="run", columns="metric", values="value", aggfunc=pm)[METRICS]
table.columns = [LABEL[c] for c in table.columns]
table.insert(0, "seeds", n_seeds)
table.loc[order]

## How wrong, and how often
All test rows of all seeds pooled. `typical miss` = median absolute error. `within ±X%` = share of listings
whose predicted price is within X % of the real price. `over-priced` = share predicted above the real price
(50 % = no bias).

In [ ]:
def load_preds(run):
    parts = []
    for seed, sd in seed_dirs(runs[run]):
        if (sd / "test_predictions.csv").exists():
            parts.append(pd.read_csv(sd / "test_predictions.csv").assign(seed=seed, run=run))
    return pd.concat(parts, ignore_index=True) if parts else None

preds = {run: q for run in order if (q := load_preds(run)) is not None}
for q in preds.values():
    q["abs_err"] = (q.predicted - q.actual).abs()
    q["ape"] = q.abs_err / q.actual * 100
    q["log_err"] = np.log(q.predicted / q.actual)

BANDS = [10, 20, 30, 50]

def error_summary(q):
    out = {"rows": len(q),
           "avg miss $": f"{q.abs_err.mean():,.0f}",
           "typical miss $": f"{q.abs_err.median():,.0f}",
           "avg miss %": round(q.ape.mean(), 1),
           "typical miss %": round(q.ape.median(), 1)}
    out |= {f"within ±{b}%": f"{(q.ape <= b).mean() * 100:.0f}%" for b in BANDS}
    out["off by >2x"] = f"{((q.predicted > 2 * q.actual) | (q.predicted < q.actual / 2)).mean() * 100:.1f}%"
    out["over-priced"] = f"{(q.predicted > q.actual).mean() * 100:.0f}%"
    return out

if not preds:
    print("no test_predictions.csv found - retrain with the current train.py")
else:
    print("All runs")
    display(pd.DataFrame({run: error_summary(q) for run, q in preds.items()}).T)
    q = preds.get(FOCUS)
    if q is not None:
        print(f"\n{FOCUS} by property type")
        display(pd.DataFrame({t: error_summary(g) for t, g in q.groupby("property_type")}).T
                .sort_values("rows", ascending=False))

Cumulative error curve: for any error size on the x axis, the y value is the share of listings the model
gets at least that close. Higher and further left is better. Read off e.g. "70 % of listings are within ±30 %".

In [ ]:
if preds:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    x = np.linspace(0, 100, 201)
    shown = [r for r in order if r in preds][:8]
    ax = axes[0]
    for k, run in enumerate(shown):
        a = np.sort(preds[run].ape.values)
        ax.plot(x, np.searchsorted(a, x, side="right") / len(a) * 100, color=SERIES[k], label=run)
    for b in BANDS[:3]:
        ax.axvline(b, color=AXIS, linewidth=1, linestyle=":")
    ax.set(xlim=(0, 100), ylim=(0, 100), xlabel="absolute error % (|predicted - actual| / actual)",
           ylabel="% of test listings within")
    ax.set_title("All runs")
    if len(shown) > 1:
        ax.legend(loc="lower right")

    ax = axes[1]
    q = preds.get(FOCUS)
    if q is not None:
        counts = q.property_type.value_counts()
        types = counts[counts >= MIN_ROWS].index.tolist()[:8]
        for k, t in enumerate(types):
            a = np.sort(q[q.property_type == t].ape.values)
            ax.plot(x, np.searchsorted(a, x, side="right") / len(a) * 100, color=SERIES[k], label=t)
        for b in BANDS[:3]:
            ax.axvline(b, color=AXIS, linewidth=1, linestyle=":")
        ax.set(xlim=(0, 100), ylim=(0, 100), xlabel="absolute error %")
        ax.set_title(f"{FOCUS} by property type")
        ax.legend(loc="lower right", ncol=2)
    fig.tight_layout()

## Overall test metrics per run
Each dot is one seed (one split); the black tick and line are the mean ± 1 std. Runs whose ranges overlap
heavily are not really different.

In [ ]:
show = ["median_ape", "rmse_log", "r2"]
fig, axes = plt.subplots(1, 3, figsize=(13, 0.45 * len(order) + 1.6), sharey=True)
for ax, m in zip(axes, show):
    for i, run in enumerate(order):
        v = overall[(overall.run == run) & (overall.metric == m)].value
        ax.scatter(v, np.full(len(v), i), s=44, color=SERIES[0], alpha=0.6, edgecolor=SURFACE, linewidth=1, zorder=3)
        sd = v.std(ddof=1) if len(v) > 1 else 0
        ax.errorbar(v.mean(), i, xerr=sd, fmt="|", markersize=14, markeredgewidth=2, color=INK,
                    elinewidth=1.5, capsize=0, zorder=4)
    ax.set_title(f"{LABEL[m]}  ({'lower' if LOWER_BETTER[m] else 'higher'} is better)")
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(range(len(order)), order)
axes[0].invert_yaxis()
fig.tight_layout()

## Is the difference real? (paired over the same seeds)
Each run minus the reference run, seed by seed, so split luck cancels out. If the error bar (std of the
per-seed differences) crosses zero, the difference is within noise.

In [ ]:
def paired(metric):
    ref = overall[(overall.run == REF) & (overall.metric == metric)].set_index("seed").value
    out = []
    for run in order:
        if run == REF:
            continue
        v = overall[(overall.run == run) & (overall.metric == metric)].set_index("seed").value
        common = ref.index.intersection(v.index).dropna()
        if len(common):
            d = v[common] - ref[common]
            out.append((run, d.mean(), d.std(ddof=1) if len(d) > 1 else 0.0, len(d)))
    return out

fig, axes = plt.subplots(1, 3, figsize=(13, 0.45 * max(len(order) - 1, 1) + 1.8), sharey=True)
for ax, m in zip(axes, show):
    res = paired(m)
    for i, (run, mu, sd, n) in enumerate(res):
        better = (mu < 0) == LOWER_BETTER[m]
        real = abs(mu) > sd and n > 1
        ax.errorbar(mu, i, xerr=sd, fmt="o", markersize=8, color=SERIES[0] if better else SERIES[1],
                    markeredgecolor=SURFACE, elinewidth=1.5, capsize=0, zorder=3)
        ax.annotate(("better" if better else "worse") + ("" if real else " · noise") + f"  (n={n})",
                    (mu, i), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8, color=INK2)
    ax.axvline(0, color=AXIS, linewidth=1.5, zorder=2)
    ax.set_title(f"Δ {LABEL[m]} vs {REF}")
    ax.grid(axis="y", visible=False)
    if res:
        ax.set_yticks(range(len(res)), [r[0] for r in res])
axes[0].invert_yaxis()
if not paired(show[0]):
    print(f"no other run shares seeds with {REF} - nothing to compare")
fig.tight_layout()

## Error by property type
Mean test MdAPE per run and property type. Darker = bigger error.

In [ ]:
by_type = test[(test.group != "overall") & (test.metric == "median_ape")]
heat = by_type.pivot_table(index="run", columns="group", values="value", aggfunc="mean").loc[order]
n_test = by_type.groupby("group").n.mean().round().astype(int)
cmap = LinearSegmentedColormap.from_list("blues", BLUES)
fig, ax = plt.subplots(figsize=(1.25 * heat.shape[1] + 3, 0.5 * len(order) + 1.6))
im = ax.imshow(heat.values, cmap=cmap, aspect="auto")
for (i, j), v in np.ndenumerate(heat.values):
    if not np.isnan(v):
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=9,
                color="#ffffff" if im.norm(v) > 0.55 else INK)
ax.set_xticks(range(heat.shape[1]), [f"{c}\nn≈{n_test[c]}" for c in heat.columns])
ax.set_yticks(range(len(order)), order)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("Test MdAPE % by property type (mean over seeds)")
fig.colorbar(im, ax=ax, shrink=0.8, label="MdAPE %")
fig.tight_layout()

## Learning curves (focus run)
Train vs validation loss per seed. The dashed line is the epoch whose weights were kept.

In [ ]:
hist = [(s, sd / "history.csv") for s, sd in seed_dirs(runs[FOCUS]) if (sd / "history.csv").exists()]
if not hist:
    print(f"{FOCUS}: no history.csv")
else:
    cols = min(len(hist), 3)
    rows_ = int(np.ceil(len(hist) / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(4.4 * cols, 3.1 * rows_), squeeze=False, sharey=True)
    for ax, (seed, f) in zip(axes.flat, hist):
        h = pd.read_csv(f)
        ax.plot(h.epoch, h.train_loss, color=SERIES[0], label="train")
        ax.plot(h.epoch, h.val_loss, color=SERIES[1], label="validation")
        best = h.loc[h.val_loss.idxmin()]
        ax.axvline(best.epoch, color=MUTED, linestyle="--", linewidth=1)
        ax.annotate(f"best epoch {int(best.epoch)}", (best.epoch, 0.04), xycoords=("data", "axes fraction"),
                    xytext=(4, 0), textcoords="offset points", fontsize=8, color=INK2)
        ax.set_title(f"seed {seed}" if seed is not None else FOCUS)
        ax.set_xlabel("epoch")
    for ax in axes.flat[len(hist):]:
        ax.set_visible(False)
    axes[0, 0].set_ylabel("loss (log price)")
    fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="upper right", ncol=2)
    fig.suptitle(f"Learning curves: {FOCUS}", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()

## Predicted vs actual (focus run, all seeds' test rows)
Log scales. Diagonal = perfect; the band is ±25 %. Points above the line are over-predicted. Types with fewer than `MIN_ROWS` test rows are skipped.

In [ ]:
p = preds.get(FOCUS)
if p is None:
    print(f"{FOCUS}: no test_predictions.csv")
else:
    counts = p.property_type.value_counts()
    types = counts[counts >= MIN_ROWS].index.tolist()
    cols = 4
    fig, axes = plt.subplots(int(np.ceil(len(types) / cols)), cols, figsize=(14, 3.4 * np.ceil(len(types) / cols)),
                             squeeze=False)
    for ax, t in zip(axes.flat, types):
        q = p[p.property_type == t]
        lo, hi = q[["actual", "predicted"]].min().min() * 0.8, q[["actual", "predicted"]].max().max() * 1.25
        line = np.array([lo, hi])
        ax.fill_between(line, line * 0.75, line * 1.25, color=GRID, alpha=0.7, linewidth=0)
        ax.plot(line, line, color=AXIS, linewidth=1.2)
        ax.scatter(q.actual, q.predicted, s=10, color=SERIES[0], alpha=0.35, linewidth=0)
        ax.set(xscale="log", yscale="log", xlim=(lo, hi), ylim=(lo, hi))
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.yaxis.set_minor_formatter(NullFormatter())
        ax.set_title(f"{t}  (n={len(q)}, MdAPE {q.ape.median():.1f}%)", fontsize=10)
    for ax in axes.flat[len(types):]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("predicted $")
    for ax in axes[-1]:
        ax.set_xlabel("actual $")
    fig.suptitle(f"Predicted vs actual: {FOCUS}", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()

## Error vs price level (all runs)
Test MdAPE within price deciles (deciles of the actual price). Shows whether a model is worse on cheap or on
expensive listings. Only the first 8 runs in the ranking are drawn.

In [ ]:
if preds:
    fig, ax = plt.subplots(figsize=(10, 4.2))
    shown = [r for r in order if r in preds][:8]
    for k, run in enumerate(shown):
        q = preds[run]
        dec = pd.qcut(q.actual, 10, duplicates="drop")
        g = q.groupby(dec, observed=True).agg(price=("actual", "median"), ape=("ape", "median"))
        ax.plot(g.price, g.ape, color=SERIES[k], marker="o", markersize=8, markeredgecolor=SURFACE, label=run)
    ax.set_xscale("log")
    ax.set_xlabel("actual price $ (median of each decile)")
    ax.set_ylabel("MdAPE %")
    ax.set_title("Test error by price decile")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=min(len(shown), 4))
    fig.tight_layout()

## Bias by property type (focus run)
log(predicted / actual) per test row. The box should sit on 0; above 0 = the model over-prices that type,
below = under-prices. Width of the box = how spread out the errors are.

In [ ]:
if p is not None:
    counts = p.property_type.value_counts()
    types = p[p.property_type.isin(counts[counts >= MIN_ROWS].index)].groupby("property_type").log_err.median().sort_values().index.tolist()
    fig, ax = plt.subplots(figsize=(10, 0.5 * len(types) + 1.4))
    ax.boxplot([p[p.property_type == t].log_err for t in types], orientation="horizontal", widths=0.55, patch_artist=True,
               showfliers=False, medianprops={"color": INK, "linewidth": 2},
               boxprops={"facecolor": BLUES[1], "edgecolor": BLUES[4]},
               whiskerprops={"color": BLUES[4]}, capprops={"color": BLUES[4]})
    ax.set_yticks(range(1, len(types) + 1), types)
    ax.axvline(0, color=AXIS, linewidth=1.5)
    ticks = np.log([0.5, 0.67, 0.8, 1, 1.25, 1.5, 2])
    ax.set_xticks(ticks, ["÷2", "÷1.5", "÷1.25", "0", "×1.25", "×1.5", "×2"])
    ax.set_xlabel("predicted / actual (outliers hidden)")
    ax.set_title(f"Over- and under-pricing by type: {FOCUS}")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()

## Worst districts (focus run)
Districts with at least 30 test rows, ranked by MdAPE. Where to look for data problems or missing features.

In [ ]:
if p is not None:
    d = (p.groupby("district").agg(rows=("ape", "size"), mdape=("ape", "median"),
                                    bias=("log_err", "median"))
         .query("rows >= 30").sort_values("mdape", ascending=False))
    d["bias"] = np.exp(d["bias"]).map(lambda x: f"×{x:.2f}")
    display(d.head(15).round(1))